# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shoaib237124/FlyRank_Internship_ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import os
import duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis + Time Window**

One row represents the daily search and analytics performance of one content page for one client on one reporting date.

For this notebook, I use March 2026 (month='2026-03') as the development slice. The internship guide recommends using a mid-panel month for feature development instead of the final month to avoid developing inside the future outcome window.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the grain
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicates
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1;
"""

grain_check = con.sql(query).df()
grain_check



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicates


In [16]:
#verify window
query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03';
"""

con.sql(query).df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature Fields**
* gsc_impressions
* gsc_clicks
* gsc_avg_position
* ga4_sessions
* ga4_engaged_sessions

These are historical measurements available before making the refresh decision.

**Label / Proxy**

This notebook does not build the final label.

For the capstone, the target will be a future observed decline or recovery defined from a later time window rather than a rule-derived label.

**Context Fields**
* report_date
* client_hash_id
* content_hash_id
* month

These identify the observation and are useful for grouping, joining, filtering, and time-aware validation.

### Excluded Fields

| Field | Why Excluded |
|-------|--------------|
| `client_hash_id` | Unique identifier; does not provide predictive information for the model. |
| `content_hash_id` | Unique identifier; does not contribute to predicting the target variable. |
| **Future outcome columns** | These contain information that would only be available after the prediction point, leading to **data leakage** and unrealistically high model performance. |




| Feature | Available when? |
|---------|------------------|
| `gsc_impressions` | Available **before** the refresh decision because it is calculated from historical Google Search Console data collected up to the current reporting date. |
| `gsc_clicks` | Available **before** the decision because it summarizes past organic clicks and does not use any future information. |
| `gsc_avg_position` | Available **before** the decision because it reflects the page's historical average search ranking at the reporting date. |
| `ga4_sessions` | Available **before** the decision when `ga4_data_available IS TRUE`, representing historical user traffic recorded before the page is reviewed. |
| `ga4_engaged_sessions` | Available **before** the decision when `ga4_data_available IS TRUE`, measuring historical user engagement without incorporating future outcomes. |

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT

report_date,
client_hash_id,
content_hash_id,

gsc_impressions,
gsc_clicks,
gsc_avg_position,
ga4_sessions,
ga4_engaged_sessions

FROM {TABLES['fact_daily']}

WHERE month='2026-03'

LIMIT 10;
"""

feature_frame = con.sql(query).df()
feature_frame



,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Grain
query = f"""
SELECT
report_date,
client_hash_id,
content_hash_id,
COUNT(*) cnt

FROM {TABLES['fact_daily']}

WHERE month='2026-03'

GROUP BY
report_date,
client_hash_id,
content_hash_id

HAVING COUNT(*)>1;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


In [19]:
# Counts & Window
query = f"""
SELECT

COUNT(*) total_rows,

MIN(report_date) first_day,

MAX(report_date) last_day

FROM {TABLES['fact_daily']}

WHERE month='2026-03';
"""

con.sql(query).df()

,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


In [20]:
# Availability
query = f"""
SELECT

COUNT(*) usable_rows

FROM {TABLES['fact_daily']}

WHERE

month='2026-03'

AND gsc_data_available IS TRUE

AND ga4_data_available IS TRUE;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,usable_rows
0,364347


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limits**

This data has several important limitations:

* History depth differs across clients, so a single calendar window may not provide the same amount of historical information for every client.
* Rows before a client's GA4 implementation are zero-filled with ga4_data_available = FALSE; these zeros do not represent true user engagement and must be filtered using the availability flag.
* This notebook analyzes only one development month (March 2026), so the findings should not be generalized to the full 17-month warehouse without additional validation.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.